# Testbench Reporting Notebook

# Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openfast_io import FileTools

pd.options.mode.copy_on_write = True
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
from reporting_helpers import (
    create_summary_table, plot_binned_data, plot_characteristic_loads,
    read_cm, plot_timeseries, stability_analysis, plot_dels_at_ws,
    report_psd_ranking, plot_psd_comparison, plot_aep,
)

# Start here, select directories
Load most summary information

In [ ]:
run_with_mpi = False   # if MPI was used to run cases, will change file structure slightly
output_folders = [
    'outputs/testbench_lite',
                 ]

labels = [os.path.basename(folder) for folder in output_folders]
# Users can also define their own labels, to be used in plots

# Select channels and load cases to compute characteristic loads
char_load_channels = ['RootMyb1','TwrBsMyt']
char_load_cases = ['1.1']

# Select channels for DELs (DLC 1.1 used for DELs)
del_channels = ['RootMyb1','TwrBsM','LSShftM']



###########################################################################################################
summary_folders = []
cm = []  # case matrices
dd = []  # dels
ss = []  # summary stats
ds = []  # del summary
char_loads = []
aep_info = []
for i_folder, output_folder in enumerate(output_folders):
    if run_with_mpi:
        output_folder = os.path.join(output_folder,'rank_0')
    summary_folders.append(os.path.join(output_folder,'iteration_0'))

    # load case matrix
    cm_i, _ = read_cm(os.path.join(output_folder,'case_matrix_combined.yaml'))
    cm.append(cm_i)
    unique_dlcs = cm_i['DLC'].unique()

    dd.append(pd.read_pickle(os.path.join(summary_folders[i_folder], 'DELs.p')))
    ssi = pd.read_pickle(os.path.join(summary_folders[i_folder], 'summary_stats.p'))
    ss.append(ssi)

    ds.append(FileTools.load_yaml(os.path.join(summary_folders[i_folder],'del_summary.yaml'), package=1))

    char_loads.append(FileTools.load_yaml(os.path.join(summary_folders[i_folder], 'characteristic_loads.yaml'), package=1))
    aep_info.append(FileTools.load_yaml(os.path.join(summary_folders[i_folder], 'aep_info.yaml')))

# TODO: allow only some info to be loaded, e.g. only DELs, or only characteristic loads


# Summary Table

In [ ]:
summ_df = create_summary_table(aep_info, char_loads, char_load_cases, char_load_channels, dd, ds, del_channels, labels)
summ_df

# Identify Any Failed Cases

In [ ]:
# Find failed cases
for controller_label, ss_i, cm_i in zip(labels, ss, cm):
    failed_ind = np.array(ss[0]['openfast_failed']['mean'] > 0)
    print(f"Failed cases ({controller_label}):")
    print(list(cm_i.iloc[failed_ind].index))

# Annual Energy Production 

In [ ]:
plot_aep(aep_info, summary_folders, ss, cm, labels)

# Damage Equivalent Loads
The DEL is calculated for each simulation, along with the lifetime DEL, weighted by the wind speed distribution.

In [ ]:
###########################################################################################################


cases = [index for index, element in enumerate(dd[0].index) if '1.1' in element]
cases = dd[0].index[cases]

dp, dws = plot_dels_at_ws(dd, ss, ds, cases=cases, channels=del_channels, labels=labels)

# Characteristic Loads
Extreme load processing:

For each mean wind speed, we take the mean of the maximum load; this is plotted below.  The maximum of these means in the characteristic load.

In [ ]:

###########################################################################################################

_,_ = plot_characteristic_loads(char_loads, char_load_cases, channels=char_load_channels, labels=labels)

# Binned data (all cases)
Binning is performed by taking the average of the channel over the specified time window.

In [ ]:
bin_channels = ['GenPwr','TwrBsMyt']

###########################################################################################################

bin_dfs = []
for summary_folder in summary_folders:
    bin_dfs.append(pd.read_pickle(os.path.join(summary_folder,'binned_all.p')))

fig, axs, dt = plot_binned_data(bin_dfs, channels=bin_channels, fig=None, labels=labels)
fig.suptitle(f'All binned data. Window = {dt} sec.')

# Binned data (by DLC)

In [ ]:
skip_binning = ['Ramp','Steady','Gust','Step']   # These timeseries are plotted, so no need to bin

for dlc in unique_dlcs:
    if dlc in skip_binning:
        continue
    bin_dfs = []
    for summary_folder in summary_folders:
        bin_dfs.append(pd.read_pickle(os.path.join(summary_folder,f'binned_dlc{dlc}.p')))

    fig, axs, dt = plot_binned_data(bin_dfs, channels=bin_channels, fig=None, labels=labels)
    fig.suptitle(f'DLC {dlc} binned data. Window = {dt} sec.')




In [ ]:

###########################################################################################################

_,_ = plot_characteristic_loads(char_loads, char_load_cases, channels=char_load_channels, labels=labels)

# Plot Stability Cases

In [ ]:

###############################################################################################################

stability_analysis(summary_folders,cm,labels)


# Timeseries Plotting
Plot the `channels` of the timeseries `cases_to_plot`

In [ ]:
cases_to_plot = ['DLC1.1_0_testbench_0'] 
channels = ['Wind1VelX','RtSpeed','GenTq', 'BldPitch1','TwrBsMyt','RootMyb1']

#######################################################################

plot_timeseries(summary_folders,cases_to_plot, channels,labels)

# Frequency Measures
The output contains the
1. N largest PSDs for each `psd_channels` and frequency domain specified in the `testbench_options.yaml`
2. PSD comparison between controllers



In [ ]:
channels_to_analyze = ['TwrBsMyt','RootMyb1','BldPitch1']
n_largest = 4
sort_index = 0  # index of the label to sort by, e.g. 0 for 'ROSCO', 1 for 'ROSCO+Boost', etc.

#############################################################################

psd_summ = []
for summary_folder in summary_folders:
    psd_summ.append(pd.read_pickle(os.path.join(summary_folder,'psd_summary.p')))

report_psd_ranking(psd_summ, labels, channels_to_analyze, n_largest)

In [ ]:
cases_to_plot = ['DLC1.1_0_testbench_0'] 
channels = ['Wind1VelX','TwrBsMyt','RootMyb1']


#############################################################################

plot_psd_comparison(summary_folders, cases_to_plot, channels)



# User Playground
Identify worst cases, compute measures using
- `ss` list of summary stats for each controller
- `dd` list of DELs for each controller

In [ ]:
# Find the cases with the largest tower loads
ss[0]['TwrBsMyt']['abs'].sort_values(ascending=False).head(10)